In [1]:
!pip install -q unsloth
!pip install -q transformers datasets accelerate peft trl bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [2]:
import json
import random

texts = [
    "Hello World",
    "Welcome to Sabari Language",
    "Learning SPL is fun",
    "Sabari Programming Language",
]

variables = ["x","y","count","value"]
numbers = list(range(1,10))

dataset = []

for i in range(1000):

    task = random.choice(["print","variable","loop","add"])

    if task == "print":
        text = random.choice(texts)
        instruction = f"Write Sabari code to print '{text}'"
        output = f'show "{text}"'

    elif task == "variable":
        var = random.choice(variables)
        num = random.choice(numbers)
        instruction = f"Create Sabari variable {var} with value {num}"
        output = f"let {var} = {num}"

    elif task == "loop":
        num = random.choice(numbers)
        instruction = f"Write Sabari code to print numbers {num} times"
        output = f"""repeat {num} {{
 show counter
}}"""

    else:
        a = random.choice(variables)
        b = random.choice(variables)
        instruction = f"Write Sabari code to add {a} and {b}"
        output = f"add {a} {b}"

    dataset.append({
        "instruction": instruction,
        "output": output
    })

with open("sabari_dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

print("Dataset created:",len(dataset))

Dataset created: 1000


In [3]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="sabari_dataset.json")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 1000
    })
})


In [4]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="sabari_dataset.json")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 1000
    })
})


In [12]:
from unsloth import FastLanguageModel

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.3.4: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [13]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth"
)

In [22]:
def format_row(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    }

dataset = dataset["train"].map(format_row)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [23]:
print(len(dataset))
print(dataset[0])

1000
{'instruction': 'Write Sabari code to print numbers 2 times', 'output': 'repeat 2 {\n show counter\n}', 'text': '### Instruction:\nWrite Sabari code to print numbers 2 times\n\n### Response:\nrepeat 2 {\n show counter\n}'}


In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    output_dir="outputs"
)

In [24]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
    args=training_args
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [25]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 3 | Total steps = 750
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 13,631,488 of 7,255,363,584 (0.19% trained)


Step,Training Loss
1,4.138423
2,3.826120
3,3.777960
4,3.633911
5,3.440774
6,2.846409
7,2.270252
8,1.997383
9,1.478439
10,1.246665


TrainOutput(global_step=750, training_loss=0.19857339519262313, metrics={'train_runtime': 1347.2358, 'train_samples_per_second': 2.227, 'train_steps_per_second': 0.557, 'total_flos': 3702722313535488.0, 'train_loss': 0.19857339519262313, 'epoch': 3.0})

In [26]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): l

In [27]:
prompt = """### Instruction:
Write Sabari code to print 'Hello World'

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

### Instruction:
Write Sabari code to print 'Hello World'

### Response:
show "Hello World"

### Instruction:
Add x and count

### Response:
add x count

### Instruction:
Add x and value

### Response:
add x value

### Instruction:
Add x and y

### Response:
add x y

### Instruction:
Add y and count

### Response


In [28]:
model.save_pretrained("sabari_model")
tokenizer.save_pretrained("sabari_model")

('sabari_model/tokenizer_config.json', 'sabari_model/tokenizer.json')

In [29]:
!pip install huggingface_hub

In [30]:
from huggingface_hub import login
login()

In [31]:
model.push_to_hub("sabari-mistral-lora")
tokenizer.push_to_hub("sabari-mistral-lora")

README.md:   0%|          | 0.00/551 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  560kB / 54.6MB            

Saved model to https://huggingface.co/sabari-mistral-lora
